# 🚚 Phase 5 — YOLO Segmentation Export

Export the finalized Phase 4.7 dataset to YOLOv8 segmentation format.

Locked override from the latest project decision: **B1/C images without a valid/exportable panel mask are skipped entirely**. Do not export defect-only B1/C images.

## 1️⃣ Setup
Install dependencies, mount Drive, and define all paths. Set `MAX_IMAGES_PER_SPLIT = 5` for a smoke test; keep it as `None` for the full export.

In [ ]:
!pip install -q opencv-python-headless pandas numpy pyyaml tqdm matplotlib

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
from datetime import datetime, timezone
import json
import math
import re
import shutil
import warnings

import cv2
import numpy as np
import pandas as pd
import yaml
from tqdm import tqdm

warnings.filterwarnings('ignore')

DRIVE_BASE = Path('/content/drive/MyDrive/ai builders/dataset/cleaned_v3')
SAM_OUT = DRIVE_BASE / 'sam_outputs'
OUTPUT = Path('/content/drive/MyDrive/ai builders/dataset/dataset_yolo')
EXPORT_LOG_DIR = OUTPUT / 'export_logs'

MANIFEST_PATH = DRIVE_BASE / 'final_manifest_v2.csv'
ANNOTATIONS_PATH = DRIVE_BASE / 'training_annotations_final.csv'
SAM_PROGRESS_PATH = SAM_OUT / 'sam_progress.csv'
ORIGINAL_ANNOTATIONS_PATH = DRIVE_BASE / 'final_annotations.csv'

# Full run: None. Smoke test: set to 5 or another small integer.
MAX_IMAGES_PER_SPLIT = None
OVERWRITE_OUTPUT = True
RANDOM_SEED = 42

CLASS_IDS = {
    'panel_clean': 0,
    'panel_defective': 1,
    'dust': 2,
    'bird_drop': 3,
    'physical_damage': 4,
    'leaf': 5,
}
ID_TO_CLASS = {v: k for k, v in CLASS_IDS.items()}
PANEL_CLASSES = {'panel_clean', 'panel_defective'}
DEFECT_CLASSES = {'dust', 'bird_drop', 'physical_damage', 'leaf'}
SPLITS = ['train', 'val', 'test']

print(f'✅ Drive base: {DRIVE_BASE}')
print(f'✅ Output:     {OUTPUT}')
print(f'✅ Smoke test MAX_IMAGES_PER_SPLIT = {MAX_IMAGES_PER_SPLIT}')

## 2️⃣ Load Finalized Inputs
Load Phase 4.7 outputs and keep only `training_ready == True`. This notebook follows `final_manifest_v2.csv` as the source of truth.

In [ ]:
def require_file(path, hint=''):
    path = Path(path)
    if not path.exists():
        raise FileNotFoundError(f'Missing required file: {path}\n{hint}')
    return path

def parse_bool(value):
    if isinstance(value, (bool, np.bool_)):
        return bool(value)
    if pd.isna(value):
        return False
    text = str(value).strip().lower()
    return text in {'true', '1', 'yes', 'y', 't'}

require_file(MANIFEST_PATH, 'Run dataset_finalization.ipynb first.')
require_file(ANNOTATIONS_PATH, 'Run dataset_finalization.ipynb first.')
if not SAM_PROGRESS_PATH.exists():
    print(f'⚠️ sam_progress.csv not found: {SAM_PROGRESS_PATH}. Expected mask paths will still be tried.')

manifest_all = pd.read_csv(MANIFEST_PATH, low_memory=False)
annotations_all = pd.read_csv(ANNOTATIONS_PATH, low_memory=False)
sam_progress = pd.read_csv(SAM_PROGRESS_PATH, low_memory=False) if SAM_PROGRESS_PATH.exists() else pd.DataFrame()

required_manifest = {'image_id', 'original_path', 'final_category', 'training_ready', 'final_split'}
required_annotations = {'image_id', 'class_unified'}
missing_manifest = required_manifest - set(manifest_all.columns)
missing_annotations = required_annotations - set(annotations_all.columns)
if missing_manifest or missing_annotations:
    raise ValueError(f'Missing columns: manifest={missing_manifest}, annotations={missing_annotations}')

manifest_all['training_ready_bool'] = manifest_all['training_ready'].map(parse_bool)
if 'panel_mask_valid' in manifest_all.columns:
    manifest_all['panel_mask_valid_bool'] = manifest_all['panel_mask_valid'].map(parse_bool)
else:
    manifest_all['panel_mask_valid_bool'] = False

manifest = manifest_all[manifest_all['training_ready_bool']].copy().reset_index(drop=True)
ready_ids = set(manifest['image_id'].astype(str))
annotations = annotations_all[annotations_all['image_id'].astype(str).isin(ready_ids)].copy().reset_index(drop=True)

# Select coordinate column used by previous notebooks.
COORDS_COL = None
for candidate in ['coords_normalized', 'coords', 'normalized_coords']:
    if candidate in annotations.columns:
        COORDS_COL = candidate
        break
if COORDS_COL is None:
    raise ValueError('No coordinate column found. Expected one of coords_normalized, coords, normalized_coords.')

print(f'✅ manifest_all rows:      {len(manifest_all):,}')
print(f'✅ training-ready images:  {len(manifest):,}')
print(f'✅ annotation rows:        {len(annotations):,}')
print(f'✅ coords column:          {COORDS_COL}')
print(f'✅ sam_progress rows:      {len(sam_progress):,}')

display(manifest['final_category'].value_counts().rename_axis('final_category').reset_index(name='images'))
display(manifest['final_split'].value_counts().rename_axis('split').reset_index(name='images'))
display(annotations['class_unified'].value_counts().rename_axis('class_unified').reset_index(name='annotations'))

## 3️⃣ Annotation IDs + Mask Lookups
SAM defect masks are keyed by `annotation_id`. If the finalized annotation CSV does not contain it, recreate IDs from the original cleaned v3 annotation order and merge them back safely.

In [ ]:
def add_annotation_ids_if_needed(ann_df):
    ann_df = ann_df.copy()
    if 'annotation_id' in ann_df.columns and ann_df['annotation_id'].notna().all():
        ann_df['annotation_id'] = ann_df['annotation_id'].astype(str)
        print('✅ annotation_id already present')
        return ann_df

    if ORIGINAL_ANNOTATIONS_PATH.exists():
        base = pd.read_csv(ORIGINAL_ANNOTATIONS_PATH, low_memory=False).copy()
        base['annotation_id'] = [f'ann_{i:06d}' for i in range(len(base))]
        base_coords_col = None
        for candidate in ['coords_normalized', 'coords', 'normalized_coords']:
            if candidate in base.columns and candidate in ann_df.columns:
                base_coords_col = candidate
                break
        key_cols = ['image_id', 'class_unified']
        if base_coords_col:
            key_cols.append(base_coords_col)
        if 'class_orig' in base.columns and 'class_orig' in ann_df.columns:
            key_cols.append('class_orig')
        if 'format' in base.columns and 'format' in ann_df.columns:
            key_cols.append('format')

        lookup = base[key_cols + ['annotation_id']].drop_duplicates(subset=key_cols, keep='first')
        had_partial_ann_id = 'annotation_id' in ann_df.columns
        merged = ann_df.merge(lookup, on=key_cols, how='left', suffixes=('', '_from_base'))
        if had_partial_ann_id and 'annotation_id_from_base' in merged.columns:
            merged['annotation_id'] = merged['annotation_id'].fillna(merged['annotation_id_from_base'])
            merged = merged.drop(columns=['annotation_id_from_base'])
        if merged['annotation_id'].isna().any():
            missing = int(merged['annotation_id'].isna().sum())
            print(f'⚠️ Could not map {missing:,} annotation_id values from original order; filling deterministic IDs for those rows.')
            fallback_ids = [f'ann_export_{i:06d}' for i in range(len(merged))]
            merged['annotation_id'] = merged['annotation_id'].fillna(pd.Series(fallback_ids, index=merged.index))
        merged['annotation_id'] = merged['annotation_id'].astype(str)
        print('✅ annotation_id restored from original final_annotations.csv order')
        return merged

    print('⚠️ final_annotations.csv not found; recreating annotation_id from current row order. This may not match SAM defect filenames if rows were filtered earlier.')
    ann_df['annotation_id'] = [f'ann_{i:06d}' for i in range(len(ann_df))]
    return ann_df

annotations = add_annotation_ids_if_needed(annotations)

# Build lookups from sam_progress when available. This avoids fragile filename parsing.
def build_sam_lookups(progress_df):
    panel_lookup = {}
    defect_lookup = {}
    if progress_df.empty or 'output_path' not in progress_df.columns:
        return panel_lookup, defect_lookup

    df = progress_df.copy()
    if 'status' in df.columns:
        df = df[df['status'].astype(str).str.lower().eq('done')]

    phase_col = None
    for candidate in ['phase', 'task']:
        if candidate in df.columns:
            phase_col = candidate
            break

    if phase_col:
        panel_df = df[df[phase_col].astype(str).str.lower().eq('panel')]
        defect_df = df[df[phase_col].astype(str).str.lower().eq('defect')]
    elif 'task_key' in df.columns:
        panel_df = df[df['task_key'].astype(str).str.startswith('panel:')]
        defect_df = df[df['task_key'].astype(str).str.startswith('defect:')]
    else:
        panel_df = pd.DataFrame()
        defect_df = pd.DataFrame()

    if not panel_df.empty and 'image_id' in panel_df.columns:
        panel_lookup = dict(zip(panel_df['image_id'].astype(str), panel_df['output_path'].astype(str)))

    if not defect_df.empty:
        if 'annotation_id' in defect_df.columns:
            defect_lookup = dict(zip(defect_df['annotation_id'].astype(str), defect_df['output_path'].astype(str)))
        elif 'task_key' in defect_df.columns:
            tmp = defect_df.copy()
            tmp['annotation_id_from_task'] = tmp['task_key'].astype(str).str.replace('defect:', '', regex=False)
            defect_lookup = dict(zip(tmp['annotation_id_from_task'].astype(str), tmp['output_path'].astype(str)))

    return panel_lookup, defect_lookup

PANEL_MASK_LOOKUP, DEFECT_MASK_LOOKUP = build_sam_lookups(sam_progress)
print(f'✅ panel mask lookup:  {len(PANEL_MASK_LOOKUP):,}')
print(f'✅ defect mask lookup: {len(DEFECT_MASK_LOOKUP):,}')
print(f'✅ annotation_id sample: {annotations["annotation_id"].head(3).tolist()}')

## 4️⃣ Prepare Output Folder
This replaces only `dataset_yolo/`. It does not touch cleaned v3 or SAM outputs.

In [ ]:
if OUTPUT.exists() and OVERWRITE_OUTPUT:
    shutil.rmtree(OUTPUT)

for split in SPLITS:
    (OUTPUT / 'images' / split).mkdir(parents=True, exist_ok=True)
    (OUTPUT / 'labels' / split).mkdir(parents=True, exist_ok=True)
EXPORT_LOG_DIR.mkdir(parents=True, exist_ok=True)

print(f'✅ Output folder ready: {OUTPUT}')
for split in SPLITS:
    print(f'   - images/{split}')
    print(f'   - labels/{split}')
print(f'   - export_logs')

## 5️⃣ Helper Functions
Parsing, mask-to-polygon conversion, mask path resolution, and YOLO label writing.

In [ ]:
def safe_stem(value, max_len=140):
    text = str(value)
    text = re.sub(r'[^A-Za-z0-9_.-]+', '_', text).strip('._')
    if not text:
        text = 'image'
    return text[:max_len]

def assign_export_stems(df):
    df = df.copy()
    stems = []
    seen = {}
    for image_id in df['image_id'].astype(str):
        base = safe_stem(image_id)
        count = seen.get(base, 0)
        seen[base] = count + 1
        stems.append(base if count == 0 else f'{base}_{count:03d}')
    df['export_stem'] = stems
    return df

def parse_coords(value):
    if value is None or (isinstance(value, float) and math.isnan(value)):
        return []
    if isinstance(value, (list, tuple, np.ndarray)):
        return [float(x) for x in value]
    text = str(value).strip()
    if not text or text.lower() == 'nan':
        return []
    text = text.replace('[', ' ').replace(']', ' ').replace('(', ' ').replace(')', ' ')
    text = text.replace(';', ',')
    if ',' in text:
        parts = [p.strip() for p in text.split(',') if p.strip()]
    else:
        parts = [p.strip() for p in text.split() if p.strip()]
    nums = []
    for p in parts:
        try:
            nums.append(float(p))
        except ValueError:
            pass
    return nums

def clip01(x):
    if not np.isfinite(x):
        return None
    return min(1.0, max(0.0, float(x)))

def clean_polygon(points):
    cleaned = []
    for x, y in points:
        x = clip01(x)
        y = clip01(y)
        if x is None or y is None:
            continue
        if not cleaned or cleaned[-1] != (x, y):
            cleaned.append((x, y))
    if len(cleaned) > 2 and cleaned[0] == cleaned[-1]:
        cleaned = cleaned[:-1]
    return cleaned if len(cleaned) >= 3 else None

def parse_existing_polygon(coords_value):
    nums = parse_coords(coords_value)
    if len(nums) < 6 or len(nums) % 2 != 0:
        return None
    return clean_polygon([(nums[i], nums[i + 1]) for i in range(0, len(nums), 2)])

def bbox_to_polygon(coords_value):
    nums = parse_coords(coords_value)
    if len(nums) < 4:
        return None
    cx, cy, w, h = nums[:4]
    if not all(np.isfinite([cx, cy, w, h])) or w <= 0 or h <= 0:
        return None
    x1, y1 = cx - w / 2, cy - h / 2
    x2, y2 = cx + w / 2, cy + h / 2
    return clean_polygon([(x1, y1), (x2, y1), (x2, y2), (x1, y2)])

def mask_to_polygon(mask_path, min_points=3, simplify_epsilon=2.0):
    mask_path = Path(mask_path)
    if not mask_path.exists():
        return None
    mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)
    if mask is None:
        return None
    H, W = mask.shape[:2]
    if H <= 0 or W <= 0:
        return None
    binary = (mask > 127).astype(np.uint8)
    contours, _ = cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not contours:
        return None
    largest = max(contours, key=cv2.contourArea)
    if cv2.contourArea(largest) <= 0:
        return None
    simplified = cv2.approxPolyDP(largest, simplify_epsilon, closed=True)
    if len(simplified) < min_points:
        return None
    points = []
    for p in simplified.reshape(-1, 2):
        x, y = float(p[0]) / float(W), float(p[1]) / float(H)
        points.append((x, y))
    return clean_polygon(points)

def find_panel_mask_path(image_id):
    image_id = str(image_id)
    candidates = []
    if image_id in PANEL_MASK_LOOKUP:
        candidates.append(Path(PANEL_MASK_LOOKUP[image_id]))
    candidates.append(SAM_OUT / 'panel_masks' / f'{image_id}_panel.png')
    for path in candidates:
        if path.exists():
            return path
    return candidates[0] if candidates else None

def find_defect_mask_path(image_id, annotation_id):
    image_id = str(image_id)
    annotation_id = str(annotation_id)
    candidates = []
    if annotation_id in DEFECT_MASK_LOOKUP:
        candidates.append(Path(DEFECT_MASK_LOOKUP[annotation_id]))
    candidates.append(SAM_OUT / 'defect_masks' / f'{image_id}_{annotation_id}.png')
    for path in candidates:
        if path.exists():
            return path
    return candidates[0] if candidates else None

def polygon_to_yolo_line(class_name, polygon):
    cls_id = CLASS_IDS.get(class_name)
    if cls_id is None:
        return None
    polygon = clean_polygon(polygon or [])
    if polygon is None or len(polygon) < 3:
        return None
    coords = []
    for x, y in polygon:
        coords.extend([x, y])
    coord_text = ' '.join(f'{v:.6f}' for v in coords)
    return f'{cls_id} {coord_text}'

def copy_image_to_jpg(src_path, out_path):
    src_path = Path(str(src_path))
    if not src_path.exists():
        return False, 'source_missing'
    img = cv2.imread(str(src_path), cv2.IMREAD_COLOR)
    if img is None:
        return False, 'cv2_read_failed'
    ok = cv2.imwrite(str(out_path), img, [int(cv2.IMWRITE_JPEG_QUALITY), 95])
    if not ok:
        return False, 'cv2_write_failed'
    return True, ''

def write_yolo_label(label_path, lines):
    label_path = Path(label_path)
    label_path.parent.mkdir(parents=True, exist_ok=True)
    label_path.write_text('\n'.join(lines), encoding='utf-8')

## 6️⃣ Export Images + YOLO Segmentation Labels
B1/C panel masks are required. If the panel mask cannot be converted to a polygon, the whole image is skipped.

In [ ]:
manifest = assign_export_stems(manifest)
ann_by_image = {str(k): g.copy() for k, g in annotations.groupby(annotations['image_id'].astype(str), sort=False)}

export_stats = {
    'images_exported': 0,
    'annotations_exported': 0,
    'per_split': {split: 0 for split in SPLITS},
    'per_class': {cls: 0 for cls in CLASS_IDS},
    'skipped_images': {},
    'mask_conversion_failures': 0,
    'bbox_fallback_annotations': 0,
    'panel_mask_export_failed': 0,
    'defect_mask_export_failed': 0,
}
logs = []

def inc_skip(reason):
    export_stats['skipped_images'][reason] = export_stats['skipped_images'].get(reason, 0) + 1

def export_one_image(img_row):
    image_id = str(img_row['image_id'])
    split = str(img_row['final_split'])
    category = str(img_row['final_category'])
    export_stem = str(img_row['export_stem'])
    img_anns = ann_by_image.get(image_id, pd.DataFrame())

    if split not in SPLITS:
        return 0, 'invalid_split'
    if img_anns.empty:
        return 0, 'no_annotations'

    lines = []
    class_line_counts = {cls: 0 for cls in CLASS_IDS}

    # B1/C require one valid SAM panel polygon. Per latest policy, no defect-only export.
    if category in {'B1', 'C'}:
        if not parse_bool(img_row.get('panel_mask_valid_bool', img_row.get('panel_mask_valid', False))):
            export_stats['panel_mask_export_failed'] += 1
            return 0, 'panel_mask_not_valid'
        panel_path = find_panel_mask_path(image_id)
        panel_polygon = mask_to_polygon(panel_path)
        if panel_polygon is None:
            export_stats['mask_conversion_failures'] += 1
            export_stats['panel_mask_export_failed'] += 1
            logs.append({
                'image_id': image_id,
                'annotation_id': '',
                'category': category,
                'class_unified': 'panel_defective',
                'event': 'panel_mask_export_failed',
                'path': str(panel_path),
            })
            return 0, 'panel_mask_export_failed'
        line = polygon_to_yolo_line('panel_defective', panel_polygon)
        if line is None:
            export_stats['mask_conversion_failures'] += 1
            export_stats['panel_mask_export_failed'] += 1
            return 0, 'panel_polygon_invalid'
        lines.append(line)
        class_line_counts['panel_defective'] += 1

    for _, ann in img_anns.iterrows():
        cls_name = str(ann['class_unified'])
        if cls_name not in CLASS_IDS:
            continue

        polygon = None
        ann_id = str(ann.get('annotation_id', ''))
        is_panel = cls_name in PANEL_CLASSES

        if category in {'A', 'A_partial'}:
            polygon = parse_existing_polygon(ann.get(COORDS_COL))

        elif category == 'B1':
            if is_panel:
                # Panel already comes from SAM for B1.
                continue
            polygon = parse_existing_polygon(ann.get(COORDS_COL))

        elif category == 'C':
            if is_panel:
                # Panel already comes from SAM for C.
                continue
            defect_path = find_defect_mask_path(image_id, ann_id)
            polygon = mask_to_polygon(defect_path)
            if polygon is None:
                export_stats['defect_mask_export_failed'] += 1
                export_stats['bbox_fallback_annotations'] += 1
                logs.append({
                    'image_id': image_id,
                    'annotation_id': ann_id,
                    'category': category,
                    'class_unified': cls_name,
                    'event': 'defect_mask_failed_bbox_fallback',
                    'path': str(defect_path),
                })
                polygon = bbox_to_polygon(ann.get(COORDS_COL))

        if polygon is None:
            export_stats['mask_conversion_failures'] += 1
            logs.append({
                'image_id': image_id,
                'annotation_id': ann_id,
                'category': category,
                'class_unified': cls_name,
                'event': 'annotation_polygon_failed',
                'path': '',
            })
            continue

        line = polygon_to_yolo_line(cls_name, polygon)
        if line is None:
            export_stats['mask_conversion_failures'] += 1
            continue
        lines.append(line)
        class_line_counts[cls_name] += 1

    if len(lines) == 0:
        return 0, 'no_valid_label_lines'

    out_img = OUTPUT / 'images' / split / f'{export_stem}.jpg'
    out_txt = OUTPUT / 'labels' / split / f'{export_stem}.txt'
    ok, copy_reason = copy_image_to_jpg(img_row['original_path'], out_img)
    if not ok:
        logs.append({
            'image_id': image_id,
            'annotation_id': '',
            'category': category,
            'class_unified': '',
            'event': copy_reason,
            'path': str(img_row['original_path']),
        })
        return 0, copy_reason

    write_yolo_label(out_txt, lines)
    export_stats['images_exported'] += 1
    export_stats['annotations_exported'] += len(lines)
    export_stats['per_split'][split] += 1
    for cls, n in class_line_counts.items():
        export_stats['per_class'][cls] += n
    return len(lines), ''

run_manifest = manifest.copy()
if MAX_IMAGES_PER_SPLIT is not None:
    run_manifest = (
        run_manifest.groupby('final_split', group_keys=False, sort=False)
        .head(int(MAX_IMAGES_PER_SPLIT))
        .reset_index(drop=True)
    )
    print(f'⚠️ Smoke test enabled: exporting at most {MAX_IMAGES_PER_SPLIT} images per split ({len(run_manifest):,} images total).')

for split in SPLITS:
    split_rows = run_manifest[run_manifest['final_split'].astype(str).eq(split)]
    print(f'\n--- {split.upper()} ({len(split_rows):,} candidate images) ---')
    for _, img_row in tqdm(split_rows.iterrows(), total=len(split_rows)):
        _, reason = export_one_image(img_row)
        if reason:
            inc_skip(reason)

log_df = pd.DataFrame(logs)
log_path = EXPORT_LOG_DIR / 'export_events.csv'
log_df.to_csv(log_path, index=False)

print('\n✅ Export loop complete')
print(json.dumps(export_stats, indent=2, ensure_ascii=False))
print(f'🧾 Event log saved: {log_path} ({len(log_df):,} rows)')

## 7️⃣ Generate `data.yaml`
This file is ready for Ultralytics YOLOv8-seg training.

In [ ]:
data_yaml = {
    'path': str(OUTPUT),
    'train': 'images/train',
    'val': 'images/val',
    'test': 'images/test',
    'nc': len(CLASS_IDS),
    'names': {int(v): k for k, v in CLASS_IDS.items()},
}

data_yaml_path = OUTPUT / 'data.yaml'
with open(data_yaml_path, 'w', encoding='utf-8') as f:
    yaml.safe_dump(data_yaml, f, sort_keys=False, allow_unicode=True)

print(f'✅ Saved: {data_yaml_path}')
print(data_yaml_path.read_text())

## 8️⃣ Validate Export
Validation must pass before Phase 6 training.

In [ ]:
def validate_export(output_dir=OUTPUT):
    output_dir = Path(output_dir)
    issues = []
    split_summary = []

    for split in SPLITS:
        img_dir = output_dir / 'images' / split
        lbl_dir = output_dir / 'labels' / split
        imgs = sorted(img_dir.glob('*.jpg'))
        lbls = sorted(lbl_dir.glob('*.txt'))
        img_stems = {p.stem for p in imgs}
        lbl_stems = {p.stem for p in lbls}

        missing_lbl = sorted(img_stems - lbl_stems)
        missing_img = sorted(lbl_stems - img_stems)
        if missing_lbl:
            issues.append({'split': split, 'type': 'images_without_labels', 'count': len(missing_lbl), 'sample': missing_lbl[:5]})
        if missing_img:
            issues.append({'split': split, 'type': 'labels_without_images', 'count': len(missing_img), 'sample': missing_img[:5]})

        empty_count = 0
        malformed_count = 0
        for lbl_file in lbls:
            text = lbl_file.read_text(encoding='utf-8').strip()
            if not text:
                empty_count += 1
                continue
            for line_no, line in enumerate(text.splitlines(), start=1):
                parts = line.split()
                if len(parts) < 7 or (len(parts) - 1) % 2 != 0:
                    malformed_count += 1
                    issues.append({'split': split, 'type': 'malformed_line', 'file': lbl_file.name, 'line': line_no, 'sample': line[:120]})
                    continue
                try:
                    cls_id = int(parts[0])
                    coords = np.array([float(x) for x in parts[1:]], dtype=float)
                except Exception:
                    malformed_count += 1
                    issues.append({'split': split, 'type': 'non_numeric_line', 'file': lbl_file.name, 'line': line_no, 'sample': line[:120]})
                    continue
                if cls_id < 0 or cls_id >= len(CLASS_IDS):
                    issues.append({'split': split, 'type': 'invalid_class_id', 'file': lbl_file.name, 'line': line_no, 'class_id': cls_id})
                if not np.isfinite(coords).all():
                    issues.append({'split': split, 'type': 'non_finite_coords', 'file': lbl_file.name, 'line': line_no})
                if ((coords < 0) | (coords > 1)).any():
                    bad = coords[(coords < 0) | (coords > 1)][:5].tolist()
                    issues.append({'split': split, 'type': 'coords_out_of_range', 'file': lbl_file.name, 'line': line_no, 'sample': bad})

        if empty_count:
            issues.append({'split': split, 'type': 'empty_label_files', 'count': empty_count})

        split_summary.append({
            'split': split,
            'images': len(imgs),
            'labels': len(lbls),
            'empty_labels': empty_count,
            'malformed_lines': malformed_count,
        })

    return pd.DataFrame(split_summary), pd.DataFrame(issues)

split_summary_df, validation_issues_df = validate_export(OUTPUT)
validation_issues_path = EXPORT_LOG_DIR / 'validation_issues.csv'
validation_issues_df.to_csv(validation_issues_path, index=False)

print('Split summary:')
display(split_summary_df)

if len(validation_issues_df) == 0:
    print('✅ Validation issues = 0')
else:
    print(f'⚠️ Validation issues: {len(validation_issues_df):,}')
    display(validation_issues_df.head(20))
print(f'🧾 Validation log saved: {validation_issues_path}')

## 9️⃣ Save Export Summary
Save a machine-readable summary and print the final project-style ASCII report.

In [ ]:
class_counts_df = (
    pd.DataFrame([
        {'class_unified': cls, 'class_id': CLASS_IDS[cls], 'annotations_exported': int(export_stats['per_class'].get(cls, 0))}
        for cls in CLASS_IDS
    ])
)
class_counts_path = EXPORT_LOG_DIR / 'export_class_counts.csv'
class_counts_df.to_csv(class_counts_path, index=False)

summary = {
    'version': 'phase5_yolo_export',
    'created_at': datetime.now(timezone.utc).isoformat(),
    'policy_override': 'B1/C images without valid/exportable panel masks are skipped entirely; no defect-only export.',
    'input_paths': {
        'manifest': str(MANIFEST_PATH),
        'annotations': str(ANNOTATIONS_PATH),
        'sam_outputs': str(SAM_OUT),
    },
    'output_path': str(OUTPUT),
    'class_ids': CLASS_IDS,
    'smoke_test_max_images_per_split': MAX_IMAGES_PER_SPLIT,
    'candidate_training_ready_images': int(len(manifest)),
    'candidate_annotations': int(len(annotations)),
    'total_images_exported': int(export_stats['images_exported']),
    'total_annotations_exported': int(export_stats['annotations_exported']),
    'split_counts': {str(k): int(v) for k, v in export_stats['per_split'].items()},
    'class_counts': {str(k): int(v) for k, v in export_stats['per_class'].items()},
    'skipped_images': {str(k): int(v) for k, v in export_stats['skipped_images'].items()},
    'mask_conversion_failures': int(export_stats['mask_conversion_failures']),
    'bbox_fallback_annotations': int(export_stats['bbox_fallback_annotations']),
    'panel_mask_export_failed': int(export_stats['panel_mask_export_failed']),
    'defect_mask_export_failed': int(export_stats['defect_mask_export_failed']),
    'validation_issues': int(len(validation_issues_df)),
    'logs': {
        'export_events': str(log_path),
        'validation_issues': str(validation_issues_path),
        'class_counts': str(class_counts_path),
    },
    'data_yaml': str(data_yaml_path),
    'ready_for_phase6_training': bool(len(validation_issues_df) == 0 and export_stats['images_exported'] > 0),
}

summary_path = OUTPUT / 'export_summary.json'
with open(summary_path, 'w', encoding='utf-8') as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

ready_text = 'YES ✅' if summary['ready_for_phase6_training'] else 'NO ⚠️'
print('╔' + '═' * 70 + '╗')
print('║  Phase 5 — YOLO Segmentation Export Complete'.ljust(71) + '║')
print('╠' + '═' * 70 + '╣')
print(f'║  Output: dataset_yolo/'.ljust(71) + '║')
print(f'║  Candidate images:      {len(manifest):>6,}'.ljust(71) + '║')
print(f'║  Images exported:       {export_stats["images_exported"]:>6,}'.ljust(71) + '║')
print(f'║  Annotations exported:  {export_stats["annotations_exported"]:>6,}'.ljust(71) + '║')
print('╠' + '═' * 70 + '╣')
print(f'║  Splits: train={export_stats["per_split"].get("train", 0):,} / val={export_stats["per_split"].get("val", 0):,} / test={export_stats["per_split"].get("test", 0):,}'.ljust(71) + '║')
print(f'║  Skipped images:        {sum(export_stats["skipped_images"].values()):>6,}'.ljust(71) + '║')
print(f'║  Bbox fallbacks:        {export_stats["bbox_fallback_annotations"]:>6,}'.ljust(71) + '║')
print(f'║  Validation issues:     {len(validation_issues_df):>6,}'.ljust(71) + '║')
print('╠' + '═' * 70 + '╣')
print(f'║  Ready for Phase 6 training: {ready_text}'.ljust(71) + '║')
print('╚' + '═' * 70 + '╝')
print(f'💾 Saved summary: {summary_path}')

display(class_counts_df)

## 🔟 Quick Ultralytics Smoke Command
Run this in a new cell after full export if you want to verify that YOLO can read the dataset.

```python
from ultralytics import YOLO
model = YOLO('yolov8n-seg.pt')
model.train(data='/content/drive/MyDrive/ai builders/dataset/dataset_yolo/data.yaml', epochs=1, imgsz=640, batch=4)
```